# Snowflake Connection Test

Quick, isolated check that a client's `.env` (under `config/clients/<client>/.env`)
can actually authenticate to Snowflake — without running the full `deploy.py`
(no stage upload, no SP/Task registration).

Set `CLIENT` below and run all cells. If auth uses browser SSO
(`SNOWFLAKE_PASSWORD` not set in the `.env`), a browser window will pop up.

In [16]:
CLIENT = "meijer"  # subfolder under config/clients/

In [17]:
import os
from pathlib import Path

# Resolve paths relative to this notebook (store_insight/pipeline/tools/)
_HERE = Path.cwd()
# Walk up until we find the store_insight package root (has pipeline/config/clients)
_PKG_DIR = _HERE
while not (_PKG_DIR / "config" / "clients").exists():
    if _PKG_DIR.parent == _PKG_DIR:
        raise RuntimeError("Could not locate store_insight/pipeline directory - run this notebook from within the repo.")
    _PKG_DIR = _PKG_DIR.parent

client_env = _PKG_DIR / "config" / "clients" / CLIENT / ".env"
root_env = _PKG_DIR / ".env"
env_path = client_env if client_env.exists() else root_env

if not env_path.exists():
    raise FileNotFoundError(f"No .env found at {client_env} or {root_env}")

print(f"Loading env from: {env_path}")

with open(env_path) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, val = line.partition("=")
            os.environ[key.strip()] = val.strip()

# Sanity print (never prints the password itself)
for k in ["SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_ROLE", "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_DATABASE", "SNOWFLAKE_SCHEMA"]:
    print(f"{k} = {os.environ.get(k)}")
print("SNOWFLAKE_PASSWORD set:", bool(os.environ.get("SNOWFLAKE_PASSWORD")))

Loading env from: c:\Users\DanielTaylor\store_insight_new_codebase\store-insight-pipeline\store_insight\pipeline\config\clients\meijer\.env
SNOWFLAKE_ACCOUNT = UUTQCJV-MEJR
SNOWFLAKE_USER = daniel.t@retailinsight.io
SNOWFLAKE_ROLE = STORE_INSIGHT_ROLE
SNOWFLAKE_WAREHOUSE = STORE_INSIGHT_WH
SNOWFLAKE_DATABASE = STORE_INSIGHT_DB
SNOWFLAKE_SCHEMA = PUBLIC
SNOWFLAKE_PASSWORD set: False


In [18]:
from snowflake.snowpark import Session

password = os.environ.get("SNOWFLAKE_PASSWORD")
params = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "role": os.environ["SNOWFLAKE_ROLE"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"],
}
if password:
    params["password"] = password
else:
    params["authenticator"] = "externalbrowser"

print("Connecting... (a browser window may open for SSO)")
session = Session.builder.configs(params).create()
print("Connected.")

Connecting... (a browser window may open for SSO)


 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZJBb9owFMe%2FSuSdk9gOTMECqgzExgSFQdrDbiZxgtvEZn4OoXz6mlCk7tAeerEs5%2Fdefvb%2FDe9OdeUdhQGp1QiRACNPqEznUpUj9JDO%2FBh5YLnKeaWVGKEXAehuPAReVweWNHavNuJfI8B6rpEC1n0YocYopjlIYIrXApjN2DZZLhgNMDsYbXWmK%2FSu5PMKDiCMdYa3khyk09tbe2Bh2LZt0EaBNmVIMcYhHoSOuiDfbvzJ3ekDnoS4d%2BEd4fD1m9sPqa5P8JnW7goB%2B5Wma3%2B92qbIS26qE62gqYXZCnOUmXjYLK4C4AzKM48JifqB4GD9Bnwa8HNjRABKt0XFn0Wm60NjXfPA7cJC5GGlS%2BnuP5%2BO0OFZ5ufV8uduMr0%2FFv2p%2BP1UPsGfcnlKZnSx3dDTPpEE8%2Bi4SgdEZ8h7vAVMLwHPARoxV5dYrTvC9LuPYx9HKcGs12eEBjTu%2FUXe1MUqFbdd5c298whqmRkNurBaVVKJznIQ8Yjk%2Fdwn3C29KCr8HefYJ1Hc7w2ynBIch5ewKboOEOtEzPiLzzIM3zd5G8l7l9J8utaVzF68mTY1tx%2BHSALSncjcLzqUiZrLKslzIwBcmFWl24kR3LrJt6YRKBxf%2F%2F

 pip install snowflake-connector-python[secure-local-storage]


In [19]:
result = session.sql(
    "SELECT CURRENT_USER() AS USER, CURRENT_ACCOUNT() AS ACCOUNT, "
    "CURRENT_ROLE() AS ROLE, CURRENT_WAREHOUSE() AS WAREHOUSE, "
    "CURRENT_DATABASE() AS DATABASE, CURRENT_SCHEMA() AS SCHEMA"
).collect()
result

[Row(USER='DANIEL.T@RETAILINSIGHT.IO', ACCOUNT='GZA81135', ROLE='STORE_INSIGHT_ROLE', WAREHOUSE='STORE_INSIGHT_WH', DATABASE='STORE_INSIGHT_DB', SCHEMA='PUBLIC')]

In [20]:
session.close()
print("Session closed.")

Session closed.
